# TCA — Environment Setup
## AudioGuard FYP | Text Context Analysis Track
**Purpose**: Run this notebook FIRST before any T1–T6 TCA notebooks. It installs all dependencies, verifies GPU/Kaggle availability, downloads the Davidson dataset, and confirms the local NLI CSV is present.

**Run order**: `00_environment_setup → T1 → T2 → T3 → T4 → T5 → T6 → T_evaluate_all`

In [ ]:
# Cell 1 — Install all TCA dependencies
# Safe to re-run — pip skips already-installed packages
%pip install transformers>=4.40.0 datasets>=2.18.0 accelerate>=0.29.0 \
             scikit-learn pandas numpy tqdm sentencepiece protobuf \
             python-dotenv kaggle torch torchvision matplotlib seaborn \
             jupyter notebook ipywidgets --quiet
print('✓ TCA dependencies installed')

In [ ]:
# Cell 2 — Verify PyTorch + CUDA
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM: {vram:.1f} GB')
else:
    print('No GPU detected — TCA small models (T1, T5) will run on CPU in ~10-20 min')
    print('T2, T4, T6 (Davidson 24k) may take 2-4 hours on CPU')
import sys, psutil, shutil
ram_gb = psutil.virtual_memory().total / 1e9
disk_gb = shutil.disk_usage('.').free / 1e9
print(f'\nPython:     {sys.version}')
print(f'RAM:        {ram_gb:.1f} GB total')
print(f'Disk free:  {disk_gb:.1f} GB')

In [ ]:
# Cell 3 — Verify Kaggle API (not needed for TCA datasets, but good to validate credentials)
import os, subprocess
kaggle_path = os.path.join(os.path.expanduser('~'), '.kaggle', 'kaggle.json')
if not os.path.exists(kaggle_path):
    print(f'⚠ kaggle.json NOT found at {kaggle_path}')
    print('TCA models do not require Kaggle, but SER models do.')
    print('To fix: https://www.kaggle.com/settings → API → Create New Token')
else:
    print(f'✓ kaggle.json found at {kaggle_path}')
    result = subprocess.run(['kaggle', 'datasets', 'list', '--max-size', '1'],
                            capture_output=True, text=True)
    if result.returncode == 0:
        print('✓ Kaggle API connection successful')
    else:
        print('✗ Kaggle API error:', result.stderr[:200])

In [ ]:
# Cell 4 — Download and cache Davidson dataset to ./datasets/davidson/
import os, json
from datasets import load_dataset

DAVIDSON_CACHE = os.path.join('..', '..', 'datasets', 'davidson')
os.makedirs(DAVIDSON_CACHE, exist_ok=True)

cache_file = os.path.join(DAVIDSON_CACHE, 'davidson_cached.json')
if os.path.exists(cache_file):
    print(f'✓ Davidson cache found at {cache_file} — skipping download')
    with open(cache_file, 'r') as f:
        preview = json.load(f)
    print(f'  Cached rows: {preview["total_rows"]}')
else:
    print('Downloading Davidson hate speech dataset from HuggingFace...')
    ds = load_dataset('tdavidson/hate_speech_offensive')
    train_df = ds['train'].to_pandas()
    print(f'✓ Davidson downloaded: {len(train_df)} rows')
    print(f'  Label distribution:\n{train_df["class"].value_counts()}')
    # Save metadata cache
    meta = {
        'total_rows': len(train_df),
        'label_counts': train_df['class'].value_counts().to_dict()
    }
    with open(cache_file, 'w') as f:
        json.dump(meta, f)
    print(f'✓ Cache metadata saved to {cache_file}')

In [ ]:
# Cell 5 — Verify NLI CSV exists
import os, pandas as pd

NLI_CSV = os.path.join('..', '..', 'datasets', 'hate_speech_ethics_dataset_300.csv')
if not os.path.exists(NLI_CSV):
    print(f'✗ NLI CSV NOT found at: {os.path.abspath(NLI_CSV)}')
    print('This file must exist at: AudioGuardMP_2026/datasets/hate_speech_ethics_dataset_300.csv')
    print('It is provided with the project — do not delete or rename it.')
else:
    df = pd.read_csv(NLI_CSV)
    print(f'✓ NLI CSV found: {os.path.abspath(NLI_CSV)}')
    print(f'  Shape: {df.shape}')
    print(f'  Columns: {list(df.columns)}')
    print(f'  Sample row:\n{df.iloc[0]}')

In [ ]:
# Cell 6 — System summary
import sys, psutil, shutil, platform
print('=== SYSTEM SUMMARY ===')
print(f'OS:         {platform.system()} {platform.version()}')
print(f'Python:     {sys.version}')
print(f'PyTorch:    {torch.__version__}')
print(f'CUDA:       {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:        {torch.cuda.get_device_name(0)}')
    print(f'VRAM:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'RAM:        {psutil.virtual_memory().total / 1e9:.1f} GB total | {psutil.virtual_memory().available / 1e9:.1f} GB free')
print(f'Disk free:  {shutil.disk_usage(".").free / 1e9:.1f} GB')
print()
print('✓ TCA environment setup complete — you may now run T1 through T6.')